In [9]:
# standard modules
import io
import json
import os
from pathlib import Path
import time

# specialized modules
import ee
import geemap
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm

# initialize the Earth Engine module.
ee.Initialize(project='trinity-438000')


# read AOI
aoi_path = Path().cwd() / 'bangladesh_aoi.geojson'




In [10]:
aoi = gpd.read_file(aoi_path).to_crs(4326)
gee_json = json.loads(aoi[['geometry']].to_json())
gee_aoi = geemap.geojson_to_ee(gee_json)

# inspect the GeoJSON as an EEObject through geemap.
test_map = geemap.Map()
test_map.centerObject(gee_aoi, 9) # Centering on the AOI instead of an undefined 'region'
test_map.addLayer(gee_aoi, {}, 'AOI')

test_map


Map(center=[22.25436727828584, 89.26850299999857], controls=(WidgetControl(options=['position', 'transparent_b…

In [11]:
# get extent
minx, miny, maxx, maxy = aoi.total_bounds

verts = [[
[minx, miny],
[minx, maxy],
[maxx, maxy],
[maxx, miny],
[minx, miny]
]]

# make normal float from np.float63
verts = [[float(x), float(y)] for x, y in verts[0]]

verts

[[89.173107, 22.184862],
 [89.173107, 22.323877],
 [89.363899, 22.323877],
 [89.363899, 22.184862],
 [89.173107, 22.184862]]

In [12]:
# date range
START = ee.Date('2026-01-13')
END = ee.Date('2026-02-13')

# date and geographic filter
col_filter = ee.Filter.And(
    ee.Filter.geometry(ee.Geometry.Polygon(verts)),
    ee.Filter.date(START, END),
)

def mask_s2_clouds(image):
    '''Masks clouds in a Sentinel-2 image using the QA band.

    Args:
        image (ee.Image): A Sentinel-2 image.

    Returns:
        ee.Image: A cloud-masked Sentinel-2 image.
    '''
    qa = image.select('QA60')

    # bits 10 and 11 are clouds and cirrus, respectively.
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    # both flags should be set to zero, indicating clear conditions.
    mask = (
        qa.bitwiseAnd(cloud_bit_mask)
        .eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    return image.updateMask(mask).divide(10000)


s2_median = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filter(col_filter)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 25))
    .filter(ee.Filter.lt('MEAN_SOLAR_ZENITH_ANGLE', 85))
    .map(mask_s2_clouds)
    .median()
)

export_params = {
    'image': s2_median.clip(gee_aoi),
    'description': 'place_in_bangladesh_2026-01and02_median',
    'folder': 'nr218',
    'fileNamePrefix': f'place_in_bangladesh_2026-01and02_median',
    'scale': 10,
    'region': gee_aoi.geometry(),
    'fileFormat': 'GeoTIFF',
    'maxPixels': 1e12
}

task = ee.batch.Export.image.toDrive(**export_params)
task.start()




In [15]:
info = task.status()
print(f"{info['description']}: {info['state']}", info.get('progress', ''))

place_in_bangladesh_2026-01and02_median: COMPLETED 
